# Dataset preparation

Preparation of the training dataset for fine-tuning eLoFTR on Sentinel-2
images.


### 1. Reads band files (.jp2) for each SAFE directory.

In [2]:
import glob
import random
from dataclasses import dataclass
from pathlib import Path

import cv2
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import rasterize
from shapely.geometry import box


BASE_DIR = Path("/kaggle/input/datasets/isaienkov/deforestation-in-ukraine")

TRAIN_SAFE_DIRS = [
    "S2A_MSIL1C_20160212T084052_N0201_R064_T36UYA_20160212T084510",
    "S2A_MSIL1C_20160621T084012_N0204_R064_T36UYA_20160621T084513",
    "S2A_MSIL1C_20160830T083602_N0204_R064_T36UYA_20160830T083600",
]

TEST_SAFE_DIRS = [
    "S2A_MSIL1C_20180919T083621_N0206_R064_T36UXA_20180919T105540",
]

DEFORESTATION_GEOJSON = BASE_DIR / "deforestation_labels.geojson"

OUTPUT_DIR = Path("/kaggle/working/")
TRAIN_OUT = OUTPUT_DIR / "train"
TEST_OUT = OUTPUT_DIR / "test"

PATCH_SIZE = 512
OVERLAP = 0.15 
STRIDE = int(PATCH_SIZE * (1 - OVERLAP))

BANDS = ["B04", "B03", "B02", "B08"]

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


### 2. Builds an RGB(+NIR) composite with identical normalization for all dates.

In [3]:
def find_granule_dir(safe_root: Path) -> Path:
    matches = glob.glob(str(safe_root / "*.SAFE" / "GRANULE" / "L1C_*"))
    if not matches:
        raise FileNotFoundError(f"Не знайдено GRANULE директорію в {safe_root}")
    return Path(matches[0])


def find_band_path(granule_dir: Path, band: str) -> Path:
    matches = glob.glob(str(granule_dir / "IMG_DATA" / f"*_{band}.jp2"))
    if not matches:
        raise FileNotFoundError(f"Не знайдено канал {band} у {granule_dir}")
    return Path(matches[0])


def find_cloud_mask_path(granule_dir: Path) -> Path | None:
    matches = glob.glob(str(granule_dir / "QI_DATA" / "MSK_CLOUDS_B00.gml"))
    return Path(matches[0]) if matches else None


def extract_tile_name(safe_dir_name: str) -> str:
    for part in safe_dir_name.split("_"):
        if part.startswith("T") and len(part) == 6 and part[1:].isalnum():
            return part[1:]
    raise ValueError(f"Не вдалось дістати назву тайлу з {safe_dir_name}")


In [4]:
def load_band(path: Path) -> np.ndarray:
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)


def get_transform_and_crs(path: Path):
    with rasterio.open(path) as src:
        return src.transform, src.crs, src.width, src.height


def percentile_stretch(band: np.ndarray, low=2, high=98) -> np.ndarray:
    """Identical normalization into [0, 255] by percentiles for all dates."""
    lo, hi = np.percentile(band[band > 0], [low, high])
    band = np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)
    return (band * 255).astype(np.uint8)


def build_composite(safe_root: Path) -> tuple[np.ndarray, rasterio.Affine, str]:
    """Returns an HxWx4 (R,G,B,NIR) uint8 composite + geotransform + crs."""
    granule_dir = find_granule_dir(safe_root)
    channels = []
    transform = crs = None
    for band in BANDS:
        band_path = find_band_path(granule_dir, band)
        arr = load_band(band_path)
        channels.append(percentile_stretch(arr))
        if transform is None:
            transform, crs, _, _ = get_transform_and_crs(band_path)
    composite = np.stack(channels, axis=-1)  # H, W, 4
    return composite, transform, str(crs)


### 3. Rasterize polygons
Rasterizes polygons from deforestation_labels.geojson and excludes (or sets aside separately) patches that intersect with deforestation zones.

In [5]:
def load_deforestation_mask(
    geojson_path: Path, tile_name: str, transform, width: int, height: int
) -> np.ndarray:
    """Rasterizes deforestation polygons for a specific tile into an HxW (bool) mask."""
    if not geojson_path.exists():
        return np.zeros((height, width), dtype=bool)

    gdf = gpd.read_file(geojson_path)
    gdf = gdf[gdf["tile"] == tile_name]
    if gdf.empty:
        return np.zeros((height, width), dtype=bool)

    shapes = [(geom, 1) for geom in gdf.geometry]
    mask = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )
    return mask.astype(bool)


### 4. Slices the image into fixed-size patches with overlap.

In [6]:
@dataclass
class Patch:
    row: int
    col: int
    image: np.ndarray 
    excluded: bool 


def extract_patches(
    composite: np.ndarray, deforest_mask: np.ndarray
) -> list[Patch]:
    h, w = composite.shape[:2]
    patches = []
    for row in range(0, h - PATCH_SIZE + 1, STRIDE):
        for col in range(0, w - PATCH_SIZE + 1, STRIDE):
            img_patch = composite[row : row + PATCH_SIZE, col : col + PATCH_SIZE]

            if img_patch[..., :3].std() < 5:
                continue

            mask_patch = deforest_mask[row : row + PATCH_SIZE, col : col + PATCH_SIZE]
            excluded = mask_patch.any()

            patches.append(Patch(row=row, col=col, image=img_patch, excluded=excluded))
    return patches


In [7]:
def random_homography(size: int, max_shift_frac: float = 0.08) -> np.ndarray:
    """Generates a random homography: small perspective warp + rotation."""
    max_shift = size * max_shift_frac
    src = np.float32([[0, 0], [size, 0], [size, size], [0, size]])
    dst = src + np.random.uniform(-max_shift, max_shift, src.shape).astype(np.float32)
    H = cv2.getPerspectiveTransform(src, dst)
    return H


def apply_homography(image: np.ndarray, H: np.ndarray) -> np.ndarray:
    size = image.shape[0]
    return cv2.warpPerspective(image, H, (size, size), borderMode=cv2.BORDER_REFLECT)


### 5. Form training pairs:
(patch date A, patch date B) — the same
geographical area of the same tile in different seasons — and additionally
applies a random homography to one of the patches in the pair to
simulate a change in viewpoint/sensor (this is how SuperPoint/LoFTR-like
models are trained when there is a lack of "natural" viewpoint changes).

In [8]:
def build_pairs_for_tile(safe_dirs: list[Path], out_dir: Path, apply_synthetic_homography: bool):
    """
    For a list of SAFE directories of the same tile (different dates):
    1. Builds composites and a patch grid (the same pixel grid for all dates,
       since Sentinel-2 tiles are geo-referenced and aligned with each other).
    2. Forms patch pairs (date_i, date_j) for all i < j.
    3. Optionally adds a synthetic homography to one of the patches in the pair.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    excluded_dir = out_dir / "excluded_deforestation"
    excluded_dir.mkdir(parents=True, exist_ok=True)

    tile_name = extract_tile_name(safe_dirs[0].name)

    all_patches = [] 
    for safe_dir in safe_dirs:
        composite, transform, _ = build_composite(safe_dir)
        deforest_mask = load_deforestation_mask(
            DEFORESTATION_GEOJSON, tile_name, transform,
            composite.shape[1], composite.shape[0],
        )
        patches = extract_patches(composite, deforest_mask)
        all_patches.append(patches)
        print(f"[{safe_dir.name}] find {len(patches)} patches "
              f"({sum(p.excluded for p in patches)} with deforestation)")

    coord_to_patch_per_date = [
        {(p.row, p.col): p for p in patches} for patches in all_patches
    ]
    common_coords = set(coord_to_patch_per_date[0].keys())
    for d in coord_to_patch_per_date[1:]:
        common_coords &= set(d.keys())

    pair_idx = 0
    for coord in sorted(common_coords):
        patches_at_coord = [d[coord] for d in coord_to_patch_per_date]

        target_dir = excluded_dir if any(p.excluded for p in patches_at_coord) else out_dir

        for i in range(len(patches_at_coord)):
            for j in range(i + 1, len(patches_at_coord)):
                img0 = patches_at_coord[i].image[..., :3] 
                img1 = patches_at_coord[j].image[..., :3]

                H = np.eye(3, dtype=np.float32)
                if apply_synthetic_homography:
                    H = random_homography(PATCH_SIZE)
                    img1 = apply_homography(img1, H)

                fname = f"pair_{tile_name}_{coord[0]}_{coord[1]}_{i}_{j}"
                np.savez_compressed(
                    target_dir / f"{fname}.npz",
                    image0=img0,
                    image1=img1,
                    homography=H,
                )
                preview = np.hstack([img0, img1])
                cv2.imwrite(str(target_dir / f"{fname}_preview.png"),
                            cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))
                pair_idx += 1

    print(f"Created {pair_idx} pairs in {out_dir} "
          f"(include with {excluded_dir.name}/ for deforestation-zones)")


### 6. Save pairs
Save pairs as .npz (image0, image1, homography) + separately as PNG for visual inspection.

In [9]:
train_dirs = [BASE_DIR / d for d in TRAIN_SAFE_DIRS]
test_dirs = [BASE_DIR / d for d in TEST_SAFE_DIRS]

print("=== Train dataset (tile 36UYA, 3 seasons) ===")
build_pairs_for_tile(train_dirs, TRAIN_OUT, apply_synthetic_homography=True)

print("\n=== Test dataset (tile 36UXA, hold-out) ===")
for safe_dir in test_dirs:
    composite, transform, _ = build_composite(safe_dir)
    tile_name = extract_tile_name(safe_dir.name)
    deforest_mask = load_deforestation_mask(
        DEFORESTATION_GEOJSON, tile_name, transform,
        composite.shape[1], composite.shape[0],
    )
    patches = extract_patches(composite, deforest_mask)
    TEST_OUT.mkdir(parents=True, exist_ok=True)
    for p in patches:
        fname = f"test_{tile_name}_{p.row}_{p.col}"
        np.savez_compressed(TEST_OUT / f"{fname}.npz", image=p.image)
    print(f"[{safe_dir.name}] saved {len(patches)} test patches")

=== Train dataset (tile 36UYA, 3 seasons) ===
[S2A_MSIL1C_20160212T084052_N0201_R064_T36UYA_20160212T084510] знайдено 625 патчів (0 з deforestation)
[S2A_MSIL1C_20160621T084012_N0204_R064_T36UYA_20160621T084513] знайдено 625 патчів (0 з deforestation)
[S2A_MSIL1C_20160830T083602_N0204_R064_T36UYA_20160830T083600] знайдено 625 патчів (0 з deforestation)
Created 1875 pairs in /kaggle/working/train (include with excluded_deforestation/ for deforestation-zones)

=== Test dataset (tile 36UXA, hold-out) ===
[S2A_MSIL1C_20180919T083621_N0206_R064_T36UXA_20180919T105540] збережено 625 тестових патчів
